# Extended news regressions — common-sample robustness

Этот notebook дополняет `02_extended_news_regressions_patch.ipynb`.

Главная правка: вложенные модели `M0`–`M3` сравниваются на одной и той же выборке внутри каждого тикера. 
Это важно, потому что `I_firm`, `I_market`, `I_sector` имеют разное покрытие по датам, и сравнение AIC/BIC/MSE на разных `nobs` некорректно.

Вход:
- `data/final_features_daily_extended_contexts.parquet`

Выход:
- `outputs/coverage_extended_news_indices.csv`
- `outputs/ols_extended_news_indices_common_sample.csv`
- `outputs/oos_extended_news_indices_common_sample.csv`
- `outputs/corr_extended_news_indices_common_sample.csv`

In [2]:
import os
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.metrics import mean_squared_error

DATA_PATH = "data/final_features_daily_extended_contexts.parquet"
OUT_DIR = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)

df = pd.read_parquet(DATA_PATH).copy()
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["ticker", "date"]).reset_index(drop=True)

# Alias: старый I_t трактуем как firm-specific индекс
if "I_firm" not in df.columns:
    if "I_t" in df.columns:
        df["I_firm"] = df["I_t"]
    else:
        raise ValueError("Expected either I_firm or I_t in dataframe.")

# Если названия индикаторов отличаются, нормализуем.
if "MACD" not in df.columns and "macd" in df.columns:
    df["MACD"] = df["macd"]
if "RSI" not in df.columns and "rsi" in df.columns:
    df["RSI"] = df["rsi"]

# Target: следующая дневная доходность

# First ensure r_log exists.
if "r_log" not in df.columns:
    if "returns" in df.columns:
        df["returns"] = pd.to_numeric(df["returns"], errors="coerce")
        df["r_log"] = np.log1p(df["returns"])
    else:
        price_candidates = ["adj_close", "Adj Close", "adjclose", "close", "Close"]
        price_col = next((c for c in price_candidates if c in df.columns), None)

        if price_col is None:
            raise ValueError(
                "Expected r_log, returns, or price column. Existing columns: "
                f"{df.columns.tolist()[:30]}"
            )

        df[price_col] = pd.to_numeric(df[price_col], errors="coerce")
        df["r_log"] = df.groupby("ticker")[price_col].transform(lambda s: np.log(s).diff())

# Then ensure forward return exists.
if "r_log_fwd1" not in df.columns:
    df["r_log_fwd1"] = df.groupby("ticker")["r_log"].shift(-1)

# Controls
if "r_log_lag1" not in df.columns:
    if "r_log" in df.columns:
        df["r_log_lag1"] = df.groupby("ticker")["r_log"].shift(1)
    else:
        raise ValueError("Expected r_log for lag construction.")

required = ["ticker", "date", "r_log_fwd1", "r_log_lag1", "RSI", "MACD",
            "I_firm", "I_market", "I_sector"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

df[required].head()

,ticker,date,r_log_fwd1,r_log_lag1,RSI,MACD,I_firm,I_market,I_sector
0,AAPL,2020-12-29,-0.008563,NaN,NaN,0.000000,-0.527073,NaN,NaN
1,AAPL,2020-12-30,-0.007732,NaN,0.000000,-0.089302,-0.122154,NaN,NaN
2,AAPL,2020-12-31,-0.025030,-0.008563,0.000000,-0.238232,NaN,NaN,NaN
3,AAPL,2021-01-04,0.012288,-0.007732,0.000000,-0.606907,0.659435,NaN,NaN
4,AAPL,2021-01-05,-0.034241,-0.025030,8.684289,-0.764590,-0.162034,NaN,NaN


## Coverage diagnostics

In [3]:
coverage_rows = []
for ticker, g in df.groupby("ticker"):
    total = len(g)
    row = {"ticker": ticker, "n_total": total}
    for col in ["I_firm", "I_market", "I_sector"]:
        row[f"n_{col}_nonmissing"] = int(g[col].notna().sum())
        row[f"share_{col}_nonmissing"] = float(g[col].notna().mean())
    row["n_common_M3"] = int(g[["r_log_fwd1", "r_log_lag1", "RSI", "MACD", "I_firm", "I_market", "I_sector"]].dropna().shape[0])
    row["share_common_M3"] = row["n_common_M3"] / total
    coverage_rows.append(row)

coverage = pd.DataFrame(coverage_rows)
coverage.to_csv(os.path.join(OUT_DIR, "coverage_extended_news_indices.csv"), index=False)
coverage

,ticker,n_total,n_I_firm_nonmissing,share_I_firm_nonmissing,n_I_market_nonmissing,share_I_market_nonmissing,n_I_sector_nonmissing,share_I_sector_nonmissing,n_common_M3,share_common_M3
0,AAPL,1256,728,0.579618,834,0.664013,911,0.725318,385,0.306529
1,XOM,1256,623,0.496019,831,0.661624,957,0.761943,331,0.263535


## Common-sample correlations

In [4]:
common_corr_rows = []
for ticker, g in df.groupby("ticker"):
    cols = ["r_log_fwd1", "d_RSI", "d_macd", "I_firm", "I_market", "I_sector"]
    # создадим d_RSI/d_macd при необходимости
    if "d_RSI" not in g.columns:
        df.loc[g.index, "d_RSI"] = g["RSI"].diff()
    if "d_macd" not in g.columns:
        df.loc[g.index, "d_macd"] = g["MACD"].diff()

for ticker, g in df.groupby("ticker"):
    common = g[["r_log_fwd1", "d_RSI", "d_macd", "I_firm", "I_market", "I_sector"]].dropna()
    for y in ["r_log_fwd1", "d_RSI", "d_macd"]:
        for x in ["I_firm", "I_market", "I_sector"]:
            common_corr_rows.append({
                "ticker": ticker,
                "y": y,
                "x": x,
                "n": len(common),
                "corr": common[y].corr(common[x]) if len(common) > 2 else np.nan,
            })

corr_common = pd.DataFrame(common_corr_rows)
corr_common.to_csv(os.path.join(OUT_DIR, "corr_extended_news_indices_common_sample.csv"), index=False)
corr_common

,ticker,y,x,n,corr
0,AAPL,r_log_fwd1,I_firm,385,0.011619
1,AAPL,r_log_fwd1,I_market,385,-0.011476
2,AAPL,r_log_fwd1,I_sector,385,-0.028472
3,AAPL,d_RSI,I_firm,385,0.016342
4,AAPL,d_RSI,I_market,385,0.191283
5,AAPL,d_RSI,I_sector,385,0.022720
6,AAPL,d_macd,I_firm,385,0.052056
7,AAPL,d_macd,I_market,385,0.181803
8,AAPL,d_macd,I_sector,385,0.094582
9,XOM,r_log_fwd1,I_firm,331,-0.007825


## OLS on common sample

In [5]:
MODELS = {
    "M0_controls_only": ["r_log_lag1", "RSI", "MACD"],
    "M1_firm": ["I_firm", "r_log_lag1", "RSI", "MACD"],
    "M2_firm_market": ["I_firm", "I_market", "r_log_lag1", "RSI", "MACD"],
    "M3_firm_market_sector": ["I_firm", "I_market", "I_sector", "r_log_lag1", "RSI", "MACD"],
}

def fit_ols_hac(data, y, xcols, maxlags=5):
    X = sm.add_constant(data[xcols], has_constant="add")
    Y = data[y]
    model = sm.OLS(Y, X).fit(cov_type="HAC", cov_kwds={"maxlags": maxlags})
    rows = []
    for term in model.params.index:
        rows.append({
            "model": None,
            "y": y,
            "term": term,
            "coef": model.params[term],
            "std_err_HAC": model.bse[term],
            "t": model.tvalues[term],
            "p_value": model.pvalues[term],
            "nobs": int(model.nobs),
            "r2": model.rsquared,
            "aic": model.aic,
            "bic": model.bic,
        })
    return rows

ols_rows = []
for ticker, g in df.groupby("ticker"):
    # common sample for all M0-M3
    all_cols = ["r_log_fwd1"] + MODELS["M3_firm_market_sector"]
    common = g.dropna(subset=all_cols).copy()
    for model_name, xcols in MODELS.items():
        rows = fit_ols_hac(common, "r_log_fwd1", xcols, maxlags=5)
        for r in rows:
            r["model"] = model_name
            r["ticker"] = ticker
        ols_rows.extend(rows)

ols_common = pd.DataFrame(ols_rows)
ols_common.to_csv(os.path.join(OUT_DIR, "ols_extended_news_indices_common_sample.csv"), index=False)
ols_common.head()

,model,y,term,coef,std_err_HAC,t,p_value,nobs,r2,aic,bic,ticker
0,M0_controls_only,r_log_fwd1,const,-0.008300,0.007136,-1.163177,0.244758,385,0.002681,-2069.954483,-2054.141509,AAPL
1,M0_controls_only,r_log_fwd1,r_log_lag1,-0.020827,0.097946,-0.212635,0.831612,385,0.002681,-2069.954483,-2054.141509,AAPL
2,M0_controls_only,r_log_fwd1,RSI,0.000138,0.000138,0.998417,0.318077,385,0.002681,-2069.954483,-2054.141509,AAPL
3,M0_controls_only,r_log_fwd1,MACD,-0.000392,0.000558,-0.702018,0.482668,385,0.002681,-2069.954483,-2054.141509,AAPL
4,M1_firm,r_log_fwd1,const,-0.008298,0.007120,-1.165412,0.243852,385,0.002795,-2067.998289,-2048.232073,AAPL


## Out-of-sample on common train/test split

In [6]:
def chronological_oos(data, y, xcols, train_frac=0.7):
    d = data.dropna(subset=[y] + xcols).sort_values("date").copy()
    split = int(len(d) * train_frac)
    train = d.iloc[:split]
    test = d.iloc[split:]
    if len(train) < 30 or len(test) < 10:
        return None
    X_train = sm.add_constant(train[xcols], has_constant="add")
    X_test = sm.add_constant(test[xcols], has_constant="add")
    model = sm.OLS(train[y], X_train).fit()
    pred = model.predict(X_test)
    return {
        "n_train": len(train),
        "n_test": len(test),
        "mse": mean_squared_error(test[y], pred),
    }

oos_rows = []
for ticker, g in df.groupby("ticker"):
    all_cols = ["r_log_fwd1"] + MODELS["M3_firm_market_sector"]
    common = g.dropna(subset=all_cols).sort_values("date").copy()
    split = int(len(common) * 0.7)
    train = common.iloc[:split]
    test = common.iloc[split:]
    if len(train) >= 30 and len(test) >= 10:
        baseline_pred = np.repeat(train["r_log_fwd1"].mean(), len(test))
        oos_rows.append({
            "ticker": ticker,
            "model": "baseline_mean",
            "n_train": len(train),
            "n_test": len(test),
            "mse": mean_squared_error(test["r_log_fwd1"], baseline_pred),
        })
    for model_name, xcols in MODELS.items():
        res = chronological_oos(common, "r_log_fwd1", xcols, train_frac=0.7)
        if res is not None:
            res.update({"ticker": ticker, "model": model_name})
            oos_rows.append(res)

oos_common = pd.DataFrame(oos_rows)
oos_common["mse_rel_to_baseline_pct"] = np.nan
for ticker in oos_common["ticker"].unique():
    base = oos_common.loc[(oos_common["ticker"] == ticker) & (oos_common["model"] == "baseline_mean"), "mse"]
    if len(base):
        base = float(base.iloc[0])
        mask = oos_common["ticker"] == ticker
        oos_common.loc[mask, "mse_rel_to_baseline_pct"] = (oos_common.loc[mask, "mse"] / base - 1) * 100

oos_common.to_csv(os.path.join(OUT_DIR, "oos_extended_news_indices_common_sample.csv"), index=False)
oos_common

,ticker,model,n_train,n_test,mse,mse_rel_to_baseline_pct
0,AAPL,baseline_mean,269,116,0.000267,0.000000
1,AAPL,M0_controls_only,269,116,0.000333,24.583027
2,AAPL,M1_firm,269,116,0.000334,24.947608
3,AAPL,M2_firm_market,269,116,0.000340,27.417552
4,AAPL,M3_firm_market_sector,269,116,0.000340,27.136557
5,XOM,baseline_mean,231,100,0.000203,0.000000
6,XOM,M0_controls_only,231,100,0.000221,8.870443
7,XOM,M1_firm,231,100,0.000223,9.708196
8,XOM,M2_firm_market,231,100,0.000226,11.356079
9,XOM,M3_firm_market_sector,231,100,0.000226,11.156515


## Distributed-lag model on common sample

In [7]:
# Для лаговой модели используем общий набор с лагами 0..3 по всем трём индексам.
dl_rows = []
K = 3

for ticker, g in df.groupby("ticker"):
    d = g.sort_values("date").copy()
    for col in ["I_firm", "I_market", "I_sector"]:
        for lag in range(1, K + 1):
            d[f"{col}_lag{lag}"] = d[col].shift(lag)
    xcols = []
    for col in ["I_firm", "I_market", "I_sector"]:
        xcols.append(col)
        xcols.extend([f"{col}_lag{lag}" for lag in range(1, K + 1)])
    xcols += ["r_log_lag1", "RSI", "MACD"]
    common = d.dropna(subset=["r_log_fwd1"] + xcols).copy()
    if len(common) < 40:
        continue
    rows = fit_ols_hac(common, "r_log_fwd1", xcols, maxlags=5)
    for r in rows:
        r["model"] = "DL_K3_extended_common_sample"
        r["ticker"] = ticker
    dl_rows.extend(rows)

dl_common = pd.DataFrame(dl_rows)
dl_common.to_csv(os.path.join(OUT_DIR, "distributed_lag_extended_news_indices_common_sample.csv"), index=False)
dl_common.head()

,model,y,term,coef,std_err_HAC,t,p_value,nobs,r2,aic,bic,ticker
0,DL_K3_extended_common_sample,r_log_fwd1,const,-0.012211,0.016914,-0.721963,0.470317,48,0.208908,-261.780848,-231.841632,AAPL
1,DL_K3_extended_common_sample,r_log_fwd1,I_firm,0.001394,0.007202,0.193508,0.846561,48,0.208908,-261.780848,-231.841632,AAPL
2,DL_K3_extended_common_sample,r_log_fwd1,I_firm_lag1,0.001444,0.005128,0.281581,0.778265,48,0.208908,-261.780848,-231.841632,AAPL
3,DL_K3_extended_common_sample,r_log_fwd1,I_firm_lag2,-0.002929,0.006192,-0.473013,0.636204,48,0.208908,-261.780848,-231.841632,AAPL
4,DL_K3_extended_common_sample,r_log_fwd1,I_firm_lag3,-0.005195,0.003623,-1.433842,0.151617,48,0.208908,-261.780848,-231.841632,AAPL
